# 08 — WCS Center Uncertainty (Pipeline, Fixed Bootstrap)

**Purpose:** Bootstrap estimate of the WCS orientation uncertainty at the image
center using the real pipeline (`extractor.wcsangle` + `extractor.platesolve`).

## Why this replaces the earlier version

The previous bootstrap had a critical bug in its retry logic:  when
`RemoteDisconnected` was raised during **polling** an existing submission,
the `_worker` function caught it as a `ConnectionError` and called
`platesolve_xylist` again from scratch — creating a **new login + new xylist
upload** on every retry.  With `_MAX_RETRIES=3`, each hung iteration could
queue up to 3 separate submissions.  Over 50 iterations this piles up
~150 submissions.  Astrometry.net accepts the uploads but stops creating
jobs: the "No Jobs / waiting to start" state seen on the web page.

**Fixes in this version**

1. **HTTP-level retry baked into the session** (`platesolve.py`):
   `RemoteDisconnected` during polling is now retried at the TCP/HTTP layer
   (GET only, never POST) without touching the solve logic.
2. **One shared session per bootstrap run:** `create_session()` logs in once;
   all iterations reuse the same authenticated session.  N submissions →
   1 login instead of N logins.
3. **Single-attempt worker:** no retry-from-scratch.  If a solve raises any
   exception or times out internally, the iteration is skipped cleanly.
4. **`KeyboardInterrupt` propagates cleanly** through `thread.join`.
5. **Shorter internal solve timeout** (120 s) + **longer inter-solve delay**
   (60 s) keep the server queue from backing up.
6. **Partial results saved** after each success so progress is not lost.

**Angle convention:** `north_angle_deg` is CCW from the +x pixel axis.
All scatter is computed with `angle_diff_deg` to handle wrap points.

In [1]:
import sys
import time
import pickle
import threading
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits as afits
from astropy.wcs import WCS
from scipy.ndimage import gaussian_filter

ROOT      = Path('..').resolve()
FITS_PATH = sorted((ROOT / 'data').glob('*.fit'))[0]
sys.path.insert(0, str(ROOT))

from extractor.stars import extract_stars
from extractor.platesolve import platesolve_xylist, wcs_summary, create_session
from extractor.wcsangle import angle_diff_deg, center_wcs_angle_metrics, WcsAngleMetrics

OUT_DIR = ROOT / 'out' / 'wcs_center_uncertainty_pipeline'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#111', 'axes.facecolor': '#111',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': '#ccc',  'ytick.color': '#ccc',
    'axes.edgecolor': '#555', 'axes.titlecolor': 'white',
    'legend.facecolor': '#1e1e1e', 'legend.edgecolor': '#555',
    'grid.color': '#333', 'grid.alpha': 0.4,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'savefig.facecolor': '#111', 'savefig.dpi': 150,
})

with afits.open(FITS_PATH) as hdul:
    image       = hdul[0].data.astype(float)
    orig_header = hdul[0].header.copy()

ny, nx = image.shape
xc, yc = (nx - 1) / 2.0, (ny - 1) / 2.0

_bg  = gaussian_filter(image.astype(np.float32), sigma=50)
_proc = np.clip(image.astype(np.float32) - _bg, 0, None)
_lo, _hi = np.percentile(_proc[np.isfinite(_proc)], [0.5, 99.5])
disp = np.arcsinh(np.clip(_proc, _lo, _hi))
_vlo, _vhi = float(np.arcsinh(_lo)), float(np.arcsinh(_hi))

print(f'Image  : {FITS_PATH.name}  ({nx} x {ny} px)')
print(f'Center : ({xc:.1f}, {yc:.1f}) px')
print(f'Out    : {OUT_DIR}')

Image  : fuji6_asi178_100_15s.fit  (3096 x 2080 px)
Center : (1547.5, 1039.5) px
Out    : C:\Users\bassd\Research\Spectra Angle\spectrangle\out\wcs_center_uncertainty_pipeline


## Source extraction

Extract stars once; result is shared with notebook 07 if the cache exists.

In [2]:
_shared_cache = ROOT / 'out' / 'wcs_uncertainty_tests' / 'sources_full.pkl'
_local_cache  = OUT_DIR / 'sources_full.pkl'
_src_cache    = _shared_cache if _shared_cache.exists() else _local_cache

if _src_cache.exists():
    with open(_src_cache, 'rb') as fh:
        all_xs, all_ys, all_fluxes = pickle.load(fh)
    print(f'Loaded {len(all_xs)} sources from {_src_cache.name}')
else:
    all_xs, all_ys, all_fluxes = extract_stars(image, max_sources=300, mask_spectra=True)
    with open(_local_cache, 'wb') as fh:
        pickle.dump((all_xs, all_ys, all_fluxes), fh)
    print(f'Extracted and cached {len(all_xs)} sources.')

HINTS_BASE = dict(
    center_ra=113.41,
    center_dec=30.41,
    radius=10.0,
    scale_lower=70.0,
    scale_upper=90.0,
    scale_units='arcsecperpix',
)

n_src_total = len(all_xs)
print(f'N_src={n_src_total}  '
      f'RA={HINTS_BASE["center_ra"]}  Dec={HINTS_BASE["center_dec"]}  '
      f'r={HINTS_BASE["radius"]}°')

Loaded 255 sources from sources_full.pkl
N_src=255  RA=113.41  Dec=30.41  r=10.0°


## Sanity check — single order-5 solve

Run one full solve with verbose output before any bootstrap.  This confirms
that the pipeline works end-to-end and that the session, xylist upload,
job creation, and `center_wcs_angle_metrics` all succeed.  The cell raises
a hard error if the solve fails so that the bootstrap is never started
against a broken pipeline.

If the result is already cached, the cell loads from cache instantly and
still verifies the metrics.

In [3]:
FID_ORDER = 5

# Prefer the cache from notebook 07 (identical solve).
_shared_fid = ROOT / 'out' / 'wcs_uncertainty_tests' / f'solve_order{FID_ORDER}_all.pkl'
_fid_cache  = OUT_DIR / f'solve_order{FID_ORDER}_fiducial.pkl'
if not _fid_cache.exists() and _shared_fid.exists():
    import shutil
    shutil.copy(_shared_fid, _fid_cache)
    print(f'Copied shared fiducial cache from {_shared_fid.name}')

print(f'=== SANITY CHECK: single order-{FID_ORDER} solve ===')

if _fid_cache.exists():
    print('[cached — skipping API call]')
    _sanity_verbose = False
else:
    print('Logging in ...')
    _sanity_http, _sanity_api = create_session(verbose=True)
    _sanity_verbose = True

t0 = time.time()
fid_result = platesolve_xylist(
    all_xs, all_ys, nx, ny,
    original_header=orig_header,
    hints=dict(**HINTS_BASE, tweak_order=FID_ORDER),
    fetch_products=True,
    save_products_dir=OUT_DIR / f'fiducial_order{FID_ORDER}',
    cache=_fid_cache,
    verbose=True,
    _http_sess=None if _fid_cache.exists() else _sanity_http,
    _api_session=None if _fid_cache.exists() else _sanity_api,
)
elapsed = time.time() - t0

if fid_result is None:
    raise RuntimeError(
        'Sanity-check solve FAILED — stop here.\n'
        'Check: API key, network, astrometry.net status, hints.'
    )

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    fid_wcs = WCS(fid_result.header)

fid_metrics: WcsAngleMetrics = center_wcs_angle_metrics(fid_wcs, image.shape)

print(f'\nSanity check PASSED  ({elapsed:.0f}s)')
print(f'  Submission : {fid_result.submission_id}')
print(f'  Job        : {fid_result.job_id}')
print(f'  Status     : {fid_result.status}')
print(f'  N submitted: {len(fid_result.detected_x)}')
print(f'  N matched  : {len(fid_result.matched_x)}')
print()
print(f'  RA         : {fid_metrics.ra_deg:.6f}°')
print(f'  Dec        : {fid_metrics.dec_deg:.6f}°')
print(f'  θ_north    : {fid_metrics.north_angle_deg:.5f}°  (CCW from +x pixel axis)')
print(f'  θ_east     : {fid_metrics.east_angle_deg:.5f}°')
print(f'  Uncertainty: {fid_metrics.north_angle_uncertainty_deg}  '
      f'(bootstrap required — see below)')
print()
print(wcs_summary(fid_result.header))

=== SANITY CHECK: single order-5 solve ===
[cached — skipping API call]
Loading cached result from solve_order5_fiducial.pkl

Sanity check PASSED  (0s)
  Submission : 14951701
  Job        : 15787638
  Status     : success
  N submitted: 255
  N matched  : 133

  RA         : 113.423610°
  Dec        : 30.416776°
  θ_north    : -106.98696°  (CCW from +x pixel axis)
  θ_east     : 162.97913°
  Uncertainty: None  (bootstrap required — see below)

  CTYPE1  : RA---TAN-SIP
  CTYPE2  : DEC--TAN-SIP
  CRVAL1  : 115.3833
  CRVAL2  : 28.102142
  CRPIX1  : 1504.7078
  CRPIX2  : 1161.2143
  CD1_1   : -0.0215211
  CD1_2   : 0.00619889
  CD2_1   : -0.00620073
  CD2_2   : -0.0215372
  SIP     : yes (order 5)


## Bootstrap configuration

**Design decisions (motivated by the bug analysis):**

- **One session for the entire bootstrap run.**  `create_session()` is called
  once here.  All iterations pass `_http_sess` / `_api_session` so there is
  only one login regardless of N_BOOT.  If the session expires (unlikely
  within a run), re-run this cell to refresh it.

- **Single-attempt worker — no retry-from-scratch.**  Network exceptions
  inside `platesolve_xylist` are caught and mark the iteration as failed;
  the worker does NOT call `platesolve_xylist` again.  Re-submitting on a
  network error creates duplicate queued submissions, which is exactly what
  caused the "No Jobs" failure.

- **`SOLVE_TIMEOUT_S`** is the timeout passed to `platesolve_xylist`
  (internal polling budget).  The thread timeout (`THREAD_TIMEOUT_S`) is
  slightly larger to catch pathological hangs where the internal loop is
  bypassed.  Both should be much shorter than the delay × N_BOOT budget.

- **`INTER_SOLVE_DELAY_S`** is applied only between *new* (non-cached) solves.
  60 s gives the server time to process each submission before the next
  arrives.  Increase to 90–120 s if "No Jobs" reappears.

- **`MIN_BOOT_SOURCES`**: skip samples below this count.  With 80% of 255
  sources this never triggers, but it is a guard against degenerate subsets.

- **Partial results saved** after each successful iteration to
  `bootstrap_partial.pkl`.  If the notebook is interrupted, re-running the
  bootstrap cell will skip already-cached individual solves.

In [4]:
N_BOOT              = 50
BOOT_FRAC           = 0.80
BOOT_ORDER          = 5
INTER_SOLVE_DELAY_S = 60    # seconds between new (non-cached) solves
SOLVE_TIMEOUT_S     = 120   # internal timeout passed to platesolve_xylist
THREAD_TIMEOUT_S    = 150   # hard kill-switch for the daemon thread
MIN_BOOT_SOURCES    = 50    # skip bootstrap sample if fewer sources

rng         = np.random.default_rng(42)
n_boot_src  = int(n_src_total * BOOT_FRAC)

boot_indices = [
    rng.choice(n_src_total, size=n_boot_src, replace=False)
    for _ in range(N_BOOT)
]

# Login once for the whole bootstrap run.
print('Logging in to nova.astrometry.net (once for all iterations) ...')
boot_http_sess, boot_api_session = create_session(verbose=True)
print(f'Session ready.')
print()
print(f'Bootstrap config')
print(f'  N_BOOT             = {N_BOOT}')
print(f'  BOOT_FRAC          = {BOOT_FRAC:.0%}  ({n_boot_src} sources per iteration)')
print(f'  BOOT_ORDER         = {BOOT_ORDER}')
print(f'  INTER_SOLVE_DELAY  = {INTER_SOLVE_DELAY_S}s  (new solves only)')
print(f'  SOLVE_TIMEOUT      = {SOLVE_TIMEOUT_S}s  (internal)')
print(f'  THREAD_TIMEOUT     = {THREAD_TIMEOUT_S}s  (wall-clock kill-switch)')


def _bootstrap_one(i, idx):
    """Run one bootstrap iteration. Returns (result_or_None, status_str).

    Runs platesolve_xylist in a daemon thread so a completely unresponsive
    server cannot block the notebook past THREAD_TIMEOUT_S.

    Critically: on ANY exception from platesolve_xylist the iteration is
    marked failed and we do NOT retry.  Retrying from scratch creates a new
    submission, which is what caused the "No Jobs" pile-up in the previous
    version of this notebook.
    """
    cache = OUT_DIR / f'solve_order{BOOT_ORDER}_bootstrap_{i:03d}.pkl'
    sub_x = all_xs[idx]
    sub_y = all_ys[idx]

    if len(sub_x) < MIN_BOOT_SOURCES:
        return None, f'too_few_sources ({len(sub_x)} < {MIN_BOOT_SOURCES})'

    if cache.exists():
        with open(cache, 'rb') as fh:
            r = pickle.load(fh)
        return r, 'cached'

    result_box = [None]
    error_box  = [None]

    def _worker():
        try:
            result_box[0] = platesolve_xylist(
                sub_x, sub_y, nx, ny,
                original_header=orig_header,
                hints=dict(**HINTS_BASE, tweak_order=BOOT_ORDER),
                fetch_products=False,
                cache=cache,
                verbose=False,
                timeout=SOLVE_TIMEOUT_S,
                _http_sess=boot_http_sess,
                _api_session=boot_api_session,
            )
        except Exception as exc:
            # Any exception is non-retryable: logging it here prevents
            # accidental re-submission of the xylist.
            error_box[0] = f'{type(exc).__name__}: {str(exc)[:120]}'

    thread = threading.Thread(target=_worker, daemon=True)
    thread.start()
    try:
        thread.join(timeout=THREAD_TIMEOUT_S)
    except KeyboardInterrupt:
        # Surface the interrupt so the notebook cell can be cancelled cleanly.
        raise

    if thread.is_alive():
        return None, 'thread_timeout'
    if error_box[0]:
        return None, f'exception: {error_box[0]}'
    r = result_box[0]
    return (r, 'ok') if r is not None else (None, 'solve_failed')


print('Bootstrap helper ready.')

Logging in to nova.astrometry.net (once for all iterations) ...
Logged in to nova.astrometry.net
Session ready.

Bootstrap config
  N_BOOT             = 50
  BOOT_FRAC          = 80%  (204 sources per iteration)
  BOOT_ORDER         = 5
  INTER_SOLVE_DELAY  = 60s  (new solves only)
  SOLVE_TIMEOUT      = 120s  (internal)
  THREAD_TIMEOUT     = 150s  (wall-clock kill-switch)
Bootstrap helper ready.


## Bootstrap loop

Per-iteration log format:
```
Bootstrap i/N  N_src=204  sub=<id>  job=<id>  θ=<deg>°  [status  elapsed s]
```
Partial results are written to `bootstrap_partial.pkl` after each success.
If the run is interrupted, restart the kernel, re-run setup + config cells,
then re-run this cell — already-cached solves load instantly.

In [6]:
boot_records = []   # populated in this run; analysis cell reads this
_partial_path = OUT_DIR / 'bootstrap_partial.pkl'

# Load any partial results from a previous interrupted run.
if _partial_path.exists():
    with open(_partial_path, 'rb') as fh:
        _prev = pickle.load(fh)
    print(f'Loaded {len(_prev)} partial results from previous run.')
else:
    _prev = []

# Index previously-completed iterations so we can skip them below.
_done_idxs = {r['idx'] for r in _prev}
boot_records = list(_prev)

print(f'Starting bootstrap ({N_BOOT} total, {len(_done_idxs)} already done).')
print()

for i, idx in enumerate(boot_indices):
    if i in _done_idxs:
        # Already in partial results — skip without logging.
        continue

    is_cached = (OUT_DIR / f'solve_order{BOOT_ORDER}_bootstrap_{i:03d}.pkl').exists()
    sub_x, sub_y = all_xs[idx], all_ys[idx]

    t0 = time.time()
    result, status = _bootstrap_one(i, idx)
    elapsed = time.time() - t0

    # Evaluate metrics.
    m = None
    if result is not None:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                _wcs = WCS(result.header)
            m = center_wcs_angle_metrics(_wcs, image.shape)
        except Exception as exc:
            status = f'metrics_error: {exc}'

    sub_id = getattr(result, 'submission_id', None) if result else None
    job_id = getattr(result, 'job_id',        None) if result else None

    record = dict(idx=i, metrics=m, status=status, elapsed_s=elapsed,
                  sub_id=sub_id, job_id=job_id)
    boot_records.append(record)

    # Log.
    _sub_str = f'sub={sub_id}' if sub_id else 'sub=?'
    _job_str = f'job={job_id}' if job_id else 'job=?'
    if m is not None:
        _tag = f'{_sub_str}  {_job_str}  θ={m.north_angle_deg:.4f}°  [{status}  {elapsed:.0f}s]'
    else:
        _tag = f'{_sub_str}  {_job_str}  SKIPPED  [{status}  {elapsed:.0f}s]'
    print(f'Bootstrap {i:02d}/{N_BOOT}  N_src={len(sub_x)}  {_tag}')

    # Save partial progress after each success.
    if m is not None:
        with open(_partial_path, 'wb') as fh:
            pickle.dump(boot_records, fh)

    # Delay between NEW solves only; cached solves are instant.
    if not is_cached and status in ('ok', 'solve_failed', 'thread_timeout') \
            and i < N_BOOT - 1:
        time.sleep(INTER_SOLVE_DELAY_S)

n_ok = sum(1 for r in boot_records if r['metrics'] is not None)
print(f'\nDone: {n_ok}/{N_BOOT} successful solves.')

Loaded 1 partial results from previous run.
Starting bootstrap (50 total, 1 already done).



KeyboardInterrupt: 

## Manual fallback — one solve at a time

Run this cell **once per sample** by hand.  Useful when batch bootstrap
is unreliable (e.g., during heavy server load) or when you want to top up
a partial bootstrap without running the full loop.

Each run:
1. Generates one fresh random 80% subset (truly random seed, not fixed).
2. Submits exactly one xylist — no retry-from-scratch.
3. Evaluates `center_wcs_angle_metrics` on the result.
4. Appends to `_manual_records` (persistent across cell re-runs in this kernel).
5. Prints updated count and current bootstrap std.

Results are **also** saved to `manual_bootstrap_results.pkl` so they survive
kernel restarts.  Load them with the snippet below the cell.

In [7]:
# ── persistent state across manual re-runs ───────────────────────────────────
_manual_path = OUT_DIR / 'manual_bootstrap_results.pkl'
if '_manual_records' not in dir() or _manual_records is None:
    if _manual_path.exists():
        with open(_manual_path, 'rb') as fh:
            _manual_records = pickle.load(fh)
        print(f'Loaded {len(_manual_records)} previous manual results.')
    else:
        _manual_records = []

# ── one fresh sample ─────────────────────────────────────────────────────────
_m_rng  = np.random.default_rng()           # truly random — no fixed seed
_m_idx  = _m_rng.choice(n_src_total, size=int(n_src_total * BOOT_FRAC), replace=False)
_m_sub_x, _m_sub_y = all_xs[_m_idx], all_ys[_m_idx]

# Unique cache file per run (timestamp-based) to avoid collisions.
_m_ts    = int(time.time())
_m_cache = OUT_DIR / f'manual_{_m_ts}.pkl'

print(f'Manual solve — N_src={len(_m_sub_x)}  cache={_m_cache.name}')

# Fresh session for this one solve (self-contained, works after kernel restart).
_m_http, _m_api = create_session(verbose=False)

_m_t0 = time.time()
_m_result = platesolve_xylist(
    _m_sub_x, _m_sub_y, nx, ny,
    original_header=orig_header,
    hints=dict(**HINTS_BASE, tweak_order=5),
    fetch_products=False,
    cache=_m_cache,
    verbose=True,
    timeout=SOLVE_TIMEOUT_S,
    _http_sess=_m_http,
    _api_session=_m_api,
)
_m_elapsed = time.time() - _m_t0

_m_rec = dict(ts=_m_ts, metrics=None, status='failed', elapsed_s=_m_elapsed,
              sub_id=None, job_id=None)

if _m_result is not None:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _m_wcs = WCS(_m_result.header)
        _m_metrics = center_wcs_angle_metrics(_m_wcs, image.shape)
        _m_rec.update(metrics=_m_metrics, status='ok',
                      sub_id=_m_result.submission_id, job_id=_m_result.job_id)
        print(f'SUCCESS  sub={_m_result.submission_id}  job={_m_result.job_id}')
        print(f'  θ_north = {_m_metrics.north_angle_deg:.5f}°')
    except Exception as exc:
        _m_rec['status'] = f'metrics_error: {exc}'
        print(f'Solve OK but metrics failed: {exc}')
else:
    print(f'FAILED  ({_m_elapsed:.0f}s)')

_manual_records.append(_m_rec)
with open(_manual_path, 'wb') as fh:
    pickle.dump(_manual_records, fh)

# Updated summary.
_m_ok = [r for r in _manual_records if r.get('metrics') is not None]
print()
if len(_m_ok) >= 2:
    _m_thetas = np.array([r['metrics'].north_angle_deg for r in _m_ok])
    _m_ref    = _m_thetas.mean()
    _m_std    = float(angle_diff_deg(_m_thetas, _m_ref).std())
    print(f'Manual bootstrap: N_ok={len(_m_ok)}  σ(θ_north) = {_m_std:.5f}°')
else:
    print(f'Manual bootstrap: N_ok={len(_m_ok)}  (need ≥ 2 for std)')

Manual solve — N_src=204  cache=manual_1779226638.pkl
Submitting 204 sources  (3096x2080 frame).
Source list uploaded (submission 15076715)
Waiting for job assignment....................... timed out.
FAILED  (124s)

Manual bootstrap: N_ok=0  (need ≥ 2 for std)


In [ ]:
# ── Load manual results after kernel restart ──────────────────────────────────
# Run this cell to reload _manual_records from disk without re-running the fallback.
_manual_path = OUT_DIR / 'manual_bootstrap_results.pkl'
if _manual_path.exists():
    with open(_manual_path, 'rb') as fh:
        _manual_records = pickle.load(fh)
    _m_ok = [r for r in _manual_records if r.get('metrics') is not None]
    print(f'Loaded {len(_manual_records)} manual records  ({len(_m_ok)} successful)')
    if len(_m_ok) >= 2:
        _m_thetas = np.array([r['metrics'].north_angle_deg for r in _m_ok])
        _m_ref    = _m_thetas.mean()
        _m_std    = float(angle_diff_deg(_m_thetas, _m_ref).std())
        print(f'  σ(θ_north) = {_m_std:.5f}°  (manual bootstrap)')
else:
    print('No manual results file yet.')

## Analysis — bootstrap statistics and plots

All scatter uses `angle_diff_deg(boot_θ, fiducial_θ)` — wrap-safe subtraction
mapped to (−180°, 180°].  This prevents artefacts if the north angle crosses
a wrap point between iterations.

In [ ]:
ok_records = [r for r in boot_records if r['metrics'] is not None]

if len(ok_records) < 2:
    raise RuntimeError(
        f'Only {len(ok_records)} successful bootstrap solves.\n'
        'Run more manual fallback iterations, then rerun this cell.'
    )

boot_north  = np.array([r['metrics'].north_angle_deg for r in ok_records])
boot_ra     = np.array([r['metrics'].ra_deg          for r in ok_records])
boot_dec    = np.array([r['metrics'].dec_deg         for r in ok_records])

fid_theta = fid_metrics.north_angle_deg
fid_ra    = fid_metrics.ra_deg
fid_dec   = fid_metrics.dec_deg

boot_dtheta  = angle_diff_deg(boot_north, fid_theta)                # wrap-safe
boot_dra_as  = angle_diff_deg(boot_ra, fid_ra) * 3600 * np.cos(np.radians(fid_dec))
boot_ddec_as = (boot_dec - fid_dec) * 3600
boot_sep_as  = np.sqrt(boot_dra_as**2 + boot_ddec_as**2)

boot_theta_std = float(boot_dtheta.std())
boot_sep_std   = float(boot_sep_as.std())
boot_ra_std    = float(boot_dra_as.std())
boot_dec_std   = float(boot_ddec_as.std())

print(f'Bootstrap statistics  (N_ok={len(ok_records)} / N={N_BOOT})')
print(f'  σ(θ_north)     : {boot_theta_std:.5f}°  ← adopted WCS orientation uncertainty')
print(f'  σ(sky sep)     : {boot_sep_std:.3f}"')
print(f'  σ(RA  ×cosδ)  : {boot_ra_std:.3f}"')
print(f'  σ(Dec)         : {boot_dec_std:.3f}"')
print(f'  θ range        : [{boot_north.min():.4f}°, {boot_north.max():.4f}°]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(boot_north, bins=min(15, len(ok_records)), color='#ff6b6b', edgecolor='#222', alpha=0.85)
ax.axvline(fid_theta, color='yellow', lw=2, linestyle='--', label='fiducial')
ax.axvline(boot_north.mean(), color='cyan', lw=1.5, label='boot mean')
ax.axvspan(fid_theta - boot_theta_std, fid_theta + boot_theta_std,
           alpha=0.15, color='cyan', label=f'±1σ = {boot_theta_std:.4f}°')
ax.set_xlabel('Center north angle  (°  CCW from +x)')
ax.set_ylabel('Count')
ax.set_title(f'Bootstrap θ_north  (N_ok={len(ok_records)})')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax2 = axes[1]
sc = ax2.scatter(boot_dra_as, boot_ddec_as,
                 c=boot_north, cmap='plasma', s=35, alpha=0.85, zorder=3)
plt.colorbar(sc, ax=ax2, label='θ_north  (°)', shrink=0.85)
_t = np.linspace(0, 2 * np.pi, 300)
for _r in [boot_sep_std, 2 * boot_sep_std]:
    ax2.plot(_r * np.cos(_t), _r * np.sin(_t), '--', color='cyan', lw=0.9, alpha=0.5)
ax2.axhline(0, color='#555', lw=0.7)
ax2.axvline(0, color='#555', lw=0.7)
ax2.scatter([0], [0], marker='+', color='yellow', s=150, zorder=5, label='fiducial')
ax2.set_xlabel('ΔRA × cosδ  (arcsec)')
ax2.set_ylabel('ΔDec  (arcsec)')
ax2.set_title('Bootstrap sky scatter  (colour = θ_north)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

fig.suptitle(
    f'Bootstrap WCS uncertainty  '
    f'(N={N_BOOT}, {int(BOOT_FRAC*100)}% subsets, order {BOOT_ORDER})',
    fontsize=12,
)
fig.tight_layout()
_out = OUT_DIR / 'bootstrap_north_angle.png'
fig.savefig(_out, bbox_inches='tight')
plt.show()
print(f'Saved {_out.name}')

In [ ]:
_sep = '─' * 62
print(_sep)
print('WCS CENTER ORIENTATION — PIPELINE SUMMARY')
print(_sep)
print(f'  Image            : {FITS_PATH.name}')
print(f'  Image size       : {nx} × {ny} px')
print(f'  Center pixel     : ({fid_metrics.x_center:.1f}, {fid_metrics.y_center:.1f})')
print()
print('  Fiducial WCS  (order 5, all sources)')
print(f'    RA             : {fid_metrics.ra_deg:.6f}°')
print(f'    Dec            : {fid_metrics.dec_deg:.6f}°')
print(f'    θ_north        : {fid_metrics.north_angle_deg:.5f}°  (CCW from +x pixel axis)')
print(f'    θ_east         : {fid_metrics.east_angle_deg:.5f}°')
print()
print(f'  Bootstrap  (N_ok={len(ok_records)}/{N_BOOT}, {int(BOOT_FRAC*100)}% subsets, order {BOOT_ORDER})')
print(f'    σ(θ_north)     : {boot_theta_std:.5f}°   ← adopted uncertainty')
print(f'    σ(sky sep)     : {boot_sep_std:.3f}"')
print(f'    σ(RA  ×cosδ)  : {boot_ra_std:.3f}"')
print(f'    σ(Dec)         : {boot_dec_std:.3f}"')
print(_sep)
print()
print('Interpretation')
print('  θ_north is the image-plane angle of celestial north at the image center.')
print('  It is the WCS orientation term X in the polarization-angle error budget.')
print('  σ(θ_north) is the 1σ contribution of WCS fitting uncertainty to that term.')
print('  Lens distortion enters through the WCS fit; it is not corrected separately.')